# Notebook 11 — Ensemble of B03 + B04, post-processing re-tune, submission (no training)
Solar Filament Segmentation Challenge 2026 — Kaggle edition

**Evidence so far.** B03 (ResNet-34 U-Net) and B04 (ResNet-50 U-Net) are two different, strong models that landed at the SAME validation score
(official protocol, recipe: 0.4333 vs 0.4329) and the SAME leaderboard score (0.33). A deeper encoder did not help, but two different models that make
different errors can still be **averaged** (probability averaging), which usually helps with noisy labels.

**This notebook (no training):**
1. loads the BN-fixed weights of B03 and B04 (best and last epoch of each),
2. scores each model and the ensembles under the **official protocol** (every annotator record scored separately, pooled) with the current recipe,
3. re-tunes the post-processing (threshold / minimum size / closing) on the best candidate with the split-half protocol — it only leaves the current recipe if the new setting
   wins by at least 0.005 on BOTH halves,
4. writes `submission_E01_ensemble.csv` for the 180 test images.

**Setup:** Add Input -> the competition AND the finished outputs of **B03** (folder `B03_resnet34_unet`) AND **B04** (folder `B04_resnet50_s0`). GPU on. Internet not needed. Save & Run All (~30 min).

In [1]:
# ============================================================
# CONFIGURATION
# ============================================================
KAGGLE_DATA_CANDIDATES = [
    "/kaggle/input/competitions/filament-segmentation-2026",
    "/kaggle/input/filament-segmentation-2026",
]

ENCODER = "resnet34"                    # default for the shared model cell (members choose their own)
TRAIN_SEED_OFFSET = 0                   # 0, 1, 2 ... (split is NOT affected)
assert ENCODER in ("resnet34", "resnet50", "resnet101")
EXPERIMENT = "E01_ensemble"
RUN_MODE = "FULL"                       # names kept for the shared helper cells
DEBUG_MODE = False
OUTPUT_DIR = "/kaggle/working"
SEED = 42
RESUME = True

CFG = {
    "experiment_id": EXPERIMENT,
    "train_resolution": 1024,
    "batch_size": 4,                    # if the preflight reports CUDA out-of-memory, set 2 here and re-run
    "epochs": 50,
    "lr": 1e-3,                         # decoder learning rate
    "lr_encoder": 1e-4,                 # pretrained encoder learning rate
    "weight_decay": 1e-5,
    "model": f"{ENCODER}-UNet(ImageNet)",
    "encoder": ENCODER,
    "train_seed_offset": TRAIN_SEED_OFFSET,
    "loss": "BCE+Dice",
    "prob_threshold": 0.5,
    "min_instance_px": 0,
    "debug_n_train": 24,
    "debug_n_val": 8,
    "use_amp": True,
    "num_workers": None,
    "pq_eval_every": 10,
    "snapshot_every": 10,
    "ckpt_every": 2,
    "fsync_checkpoints": False,
    "multi_record": True,               # random annotator record per training sample (B02a)
    "cosine_lr": True,                  # cosine decay to 1% of the base learning rates
    "eta_min": 1e-5,                    # decoder end LR (encoder end LR scales proportionally)
    "bn_recal_batches": 60,
    "recipe": {"tta": True, "threshold": 0.5, "min_px": 400, "closing_r": 10},
}
print(f"EXPERIMENT={EXPERIMENT} | pretrained {ENCODER} encoder | seed offset {TRAIN_SEED_OFFSET} | multi_record={CFG['multi_record']} | cosine_lr={CFG['cosine_lr']} | epochs={CFG['epochs']}")

EXPERIMENT=E01_ensemble | pretrained resnet34 encoder | seed offset 0 | multi_record=True | cosine_lr=True | epochs=50


In [2]:
import copy
import os, sys, json, random, re, warnings, math, shutil, time, hashlib, platform, gc, datetime
from pathlib import Path
from collections import defaultdict, Counter

try:
    import pycocotools
except ImportError:
    !pip -q install pycocotools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm
from pycocotools import mask as mask_utils

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ---- GPU is REQUIRED. No silent CPU fallback. ----
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. This notebook will NOT fall back to CPU.\n"
        "Fix: in the Kaggle right-hand panel open Session options -> Accelerator -> choose GPU T4 x2 or P100, "
        "then run again.")
DEVICE = torch.device("cuda:0")
N_GPUS_VISIBLE = torch.cuda.device_count()
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | device {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)} | visible GPUs: {N_GPUS_VISIBLE} "
      f"| this notebook uses ONLY cuda:0 (no DataParallel)")

if CFG["num_workers"] is None:
    CFG["num_workers"] = min(4, os.cpu_count() or 2)

# ---- output location (Kaggle working dir) ----
OUT_ROOT = Path(OUTPUT_DIR); OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT = OUT_ROOT / EXPERIMENT
OUT.mkdir(parents=True, exist_ok=True)
IN_KAGGLE = Path("/kaggle/input").exists()
print("Running on Kaggle:", IN_KAGGLE, "| Outputs:", OUT)
print("NOTE: /kaggle/working is kept only when you Save Version (Save & Run All); lost when the session ends otherwise.")

# ---- bring a previous version's checkpoints into /kaggle/working (resume across sessions) ----
RESUME_FILES = ["latest_checkpoint.pth", "latest_checkpoint_prev.pth", "baseline_best.pth",
                "baseline_best_meta.json", "training_history.csv", "split_metadata.csv"]

def import_previous_output():
    if (OUT / "latest_checkpoint.pth").exists() or not IN_KAGGLE:
        return
    hits = [p for p in Path("/kaggle/input").rglob("latest_checkpoint*.pth") if p.parent.name == OUT.name]
    if not hits:
        return
    src = max(hits, key=lambda p: p.stat().st_mtime).parent
    print(f">>> Found a previous run's checkpoints in {src} — copying them into {OUT} for resume")
    for f in RESUME_FILES + [p.name for p in src.glob("checkpoint_epoch_*.pth")]:
        if (src / f).exists() and not (OUT / f).exists():
            shutil.copyfile(src / f, OUT / f)

import_previous_output()
print("DataLoader workers:", CFG["num_workers"])

Python 3.13.16 | torch 2.11.0+cu130 | device cuda:0
GPU: Tesla T4 | visible GPUs: 2 | this notebook uses ONLY cuda:0 (no DataParallel)
Running on Kaggle: True | Outputs: /kaggle/working/E01_ensemble
NOTE: /kaggle/working is kept only when you Save Version (Save & Run All); lost when the session ends otherwise.
DataLoader workers: 4


In [3]:
# ============================================================
# DATA DISCOVERY on Kaggle (/kaggle/input) — same discovery logic as Notebook 03
# ============================================================

def acquire_data_root():
    for c in KAGGLE_DATA_CANDIDATES:
        if Path(c).exists():
            return Path(c)
    base = Path("/kaggle/input")
    if base.exists() and any(base.rglob("train_images")):
        print("WARNING: standard path not found; using /kaggle/input and auto-discovery.")
        return base
    raise FileNotFoundError(
        "Competition data not found under /kaggle/input.\n"
        "Fix: right-hand panel -> Add Input -> search 'filament-segmentation-2026' -> Add.")

DATA_ROOT = acquire_data_root()
print("DATA_ROOT:", DATA_ROOT)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def discover_dataset(root):
    """Find train/test image dirs and the COCO annotation JSON. Fail loudly."""
    img_dirs = []
    for d in sorted({p.parent for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS}):
        n = sum(1 for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)
        img_dirs.append((d, n))
        print(f"  image dir: {d.relative_to(root)}  ({n} images)")
    train_cands = [d for d, _ in img_dirs
                   if "train" in str(d).lower() and "test" not in str(d).lower()]
    test_cands = [d for d, _ in img_dirs
                  if "test" in str(d).lower() and "train" not in str(d).lower()]
    if len(train_cands) != 1 or len(test_cands) != 1:
        raise RuntimeError(
            f"Ambiguous dataset layout. train candidates={train_cands}, "
            f"test candidates={test_cands}. Inspect the listing above.")
    coco = None
    for p in root.rglob("*.json"):
        try:
            with open(p) as f:
                d = json.load(f)
            if isinstance(d, dict) and "images" in d and "annotations" in d:
                coco = p; break
        except Exception:
            pass
    if coco is None:
        raise RuntimeError(f"No COCO-style annotation JSON found under {root}")
    return train_cands[0], test_cands[0], coco

TRAIN_DIR, TEST_DIR, ANN_PATH = discover_dataset(DATA_ROOT)

train_paths = sorted(p for p in TRAIN_DIR.iterdir() if p.suffix.lower() in IMG_EXTS)
test_paths  = sorted(p for p in TEST_DIR.iterdir()  if p.suffix.lower() in IMG_EXTS)
name_to_path = {p.name: p for p in train_paths}

print(f"\ntrain images: {len(train_paths)} | test images: {len(test_paths)}")
print(f"annotations: {ANN_PATH.name}")
assert len(train_paths) > 0 and len(test_paths) > 0

DATA_ROOT: /kaggle/input/competitions/filament-segmentation-2026
  image dir: MAGFiLO_1.0_Kaggle_2026/test/test_images  (180 images)
  image dir: MAGFiLO_1.0_Kaggle_2026/train/train_images  (707 images)

train images: 707 | test images: 180
annotations: MAGFiLO_1.0_Annotations_kaggle2026_train.json


In [4]:
# ============================================================
# ANNOTATIONS + BASELINE GT REPRESENTATION  (unchanged from Notebook 03)
# Baseline target = FIRST annotation record per file (sorted by id).
# ============================================================

def load_annotations(path):
    with open(path) as f:
        coco = json.load(f)
    images_info = {im["id"]: im for im in coco["images"]}
    anns_by_image = defaultdict(list)
    for a in coco["annotations"]:
        anns_by_image[a["image_id"]].append(a)
    return coco, images_info, anns_by_image

COCO, images_info, anns_by_image = load_annotations(ANN_PATH)
print(f"JSON records: {len(images_info)} | annotations: {len(COCO['annotations'])}")

file_to_records = defaultdict(list)
for img_id, info in images_info.items():
    fname = Path(info["file_name"]).name
    if fname in name_to_path:
        file_to_records[fname].append(img_id)

missing = len(images_info) - sum(len(v) for v in file_to_records.values())
if missing:
    raise RuntimeError(f"{missing} JSON records reference files not on disk")

file_to_primary = {f: sorted(ids)[0] for f, ids in file_to_records.items()}
primary_ids = sorted(file_to_primary.values())
print(f"physical files: {len(file_to_primary)} | primary records: {len(primary_ids)}")

def ann_to_mask(ann, h, w):
    m = np.zeros((h, w), np.uint8)
    for poly in ann["segmentation"]:
        pts = np.asarray(poly, np.float64).reshape(-1, 2)
        cv2.fillPoly(m, [np.round(pts).astype(np.int32)], 1)
    return m.astype(bool)

def build_gt_masks(img_id):
    """Full-resolution instance masks for one record."""
    info = images_info[img_id]
    masks = [ann_to_mask(a, info["height"], info["width"])
             for a in anns_by_image[img_id]]
    masks = [m for m in masks if m.any()]
    if not masks:
        raise ValueError(f"Record {img_id} has no non-empty GT masks")
    return masks

def load_image(path):
    img = np.array(Image.open(path))
    if img.ndim == 3:
        img = img[..., :3].mean(axis=2)
    return img.astype(np.float32)

JSON records: 1154 | annotations: 8199
physical files: 707 | primary records: 707


In [5]:
# ============================================================
# OFFICIAL EVALUATOR REPRODUCTION
# Many-to-many matching: EVERY (pred, gt) pair with IoU > 0.5 (STRICT >)
# is a TP and contributes its IoU to the numerator.
# FP = pred with zero qualifying matches; FN = gt with zero.
# PQ = sum(TP IoUs) / (TP + 0.5 FP + 0.5 FN)
# Diagnostics (one-to-many / many-to-one) use overlap > 0.
#
# Only change vs Notebook 03: the pairwise IoU/Dice loop skips pairs whose
# bounding boxes do not intersect and only intersects the bbox overlap region.
# Results are mathematically identical (verified against the original
# implementation in the equivalence test below); it is just much faster.
# ============================================================

def _bbox(m):
    rows = np.flatnonzero(m.any(axis=1)); cols = np.flatnonzero(m.any(axis=0))
    if len(rows) == 0:
        return (0, 0, 0, 0)
    return (rows[0], rows[-1] + 1, cols[0], cols[-1] + 1)

def iou_and_dice_matrix(pred_masks, gt_masks):
    n_p, n_g = len(pred_masks), len(gt_masks)
    iou = np.zeros((n_p, n_g)); dce = np.zeros((n_p, n_g))
    pb = [_bbox(p) for p in pred_masks]; gb = [_bbox(g) for g in gt_masks]
    pa = [int(p.sum()) for p in pred_masks]; ga = [int(g.sum()) for g in gt_masks]
    for i, p in enumerate(pred_masks):
        py0, py1, px0, px1 = pb[i]
        for j, g in enumerate(gt_masks):
            gy0, gy1, gx0, gx1 = gb[j]
            y0, y1 = max(py0, gy0), min(py1, gy1)
            x0, x1 = max(px0, gx0), min(px1, gx1)
            if y0 >= y1 or x0 >= x1:
                continue
            inter = int(np.logical_and(p[y0:y1, x0:x1], g[y0:y1, x0:x1]).sum())
            if inter == 0:
                continue
            iou[i, j] = inter / (pa[i] + ga[j] - inter)
            dce[i, j] = 2 * inter / (pa[i] + ga[j])
    return iou, dce

def compute_official_pq(gt_masks, pred_masks):
    if len(gt_masks) == 0 and len(pred_masks) == 0:
        return {"PQ": 1.0, "TP": 0, "FP": 0, "FN": 0,
                "tp_ious": [], "tp_dices": [],
                "one_to_many": 0, "many_to_one": 0}
    iou, dce = iou_and_dice_matrix(pred_masks, gt_masks)
    hit = iou > 0.5                                   # STRICT >
    tp_ious = iou[hit].tolist()
    tp_dices = dce[hit].tolist()
    tp = len(tp_ious)
    fp = int((hit.sum(axis=1) == 0).sum()) if len(pred_masks) else 0
    fn = int((hit.sum(axis=0) == 0).sum()) if len(gt_masks) else 0
    denom = tp + 0.5 * fp + 0.5 * fn
    overlap = iou > 0
    one_to_many = int((overlap.sum(axis=0) > 1).sum())   # GT hit by multiple preds
    many_to_one = int((overlap.sum(axis=1) > 1).sum())   # pred hitting multiple GTs
    return {"PQ": (sum(tp_ious) / denom) if denom > 0 else 0.0,
            "TP": tp, "FP": fp, "FN": fn,
            "tp_ious": tp_ious, "tp_dices": tp_dices,
            "one_to_many": one_to_many, "many_to_one": many_to_one}

# ---------------- MANDATORY UNIT TESTS (same 7 as Notebook 03) ----------------
def _box(y0, y1, x0, x1, hw=64):
    m = np.zeros((hw, hw), bool); m[y0:y1, x0:x1] = True; return m

def run_metric_tests():
    results = []
    g = _box(10, 20, 10, 30)                       # 10x20 = 200 px

    r = compute_official_pq([g], [g.copy()])
    results.append(("T1 perfect -> PQ=1", abs(r["PQ"] - 1.0) < 1e-9))

    half = _box(10, 20, 10, 20)                    # IoU = 0.5 exactly
    r = compute_official_pq([g], [half])
    results.append(("T2 IoU==0.5 not TP", r["TP"] == 0 and r["FP"] == 1 and r["FN"] == 1))

    most = _box(10, 20, 10, 24)                    # IoU = 0.7
    r = compute_official_pq([g], [most])
    results.append(("T3 IoU>0.5 is TP", r["TP"] == 1 and r["FP"] == 0 and r["FN"] == 0))

    p1 = _box(10, 20, 10, 27); p2 = _box(10, 20, 12, 30)
    r = compute_official_pq([g], [p1, p2])
    results.append(("T4 1 GT + 2 preds -> TP=2 FP=0 FN=0",
                    r["TP"] == 2 and r["FP"] == 0 and r["FN"] == 0))

    g1 = _box(10, 20, 10, 30); g2 = _box(10, 20, 8, 32); pred = _box(10, 20, 9, 31)
    r = compute_official_pq([g1, g2], [pred])
    results.append(("T5 2 GT + 1 pred -> TP=2 FP=0 FN=0",
                    r["TP"] == 2 and r["FP"] == 0 and r["FN"] == 0))

    r = compute_official_pq([g], [])
    results.append(("T6 no preds -> FN=1, PQ=0", r["FN"] == 1 and r["PQ"] == 0.0))

    r = compute_official_pq([], [g])
    results.append(("T7 no GT -> FP=1, PQ=0", r["FP"] == 1 and r["PQ"] == 0.0))

    all_ok = True
    for name, ok in results:
        print(f"  {'PASS' if ok else 'FAIL'}  {name}")
        all_ok &= ok
    if not all_ok:
        raise AssertionError("Metric unit tests FAILED — do not proceed")
    print("All 7 metric tests passed.")

run_metric_tests()

# ---------------- EQUIVALENCE TEST: fast matrix == original Notebook-03 matrix ----------------
def _iou_and_dice_matrix_reference(pred_masks, gt_masks):
    """Verbatim original implementation from Notebook 03 (slow, full-frame)."""
    n_p, n_g = len(pred_masks), len(gt_masks)
    iou = np.zeros((n_p, n_g)); dce = np.zeros((n_p, n_g))
    for i, p in enumerate(pred_masks):
        ps = p.sum()
        for j, g in enumerate(gt_masks):
            inter = np.logical_and(p, g).sum()
            if inter == 0:
                continue
            gs = g.sum()
            iou[i, j] = inter / (ps + gs - inter)
            dce[i, j] = 2 * inter / (ps + gs)
    return iou, dce

def run_equivalence_test(n_trials=40, hw=96):
    rng = np.random.RandomState(0)
    def rand_mask():
        m = np.zeros((hw, hw), np.uint8)
        for _ in range(rng.randint(1, 4)):
            y, x = rng.randint(0, hw - 8, 2); h, w = rng.randint(2, 30, 2)
            m[y:y + h, x:x + w] = 1
        return m.astype(bool)
    for _ in range(n_trials):
        P = [rand_mask() for _ in range(rng.randint(0, 6))]
        G = [rand_mask() for _ in range(rng.randint(0, 6))]
        a = iou_and_dice_matrix(P, G); b = _iou_and_dice_matrix_reference(P, G)
        assert np.allclose(a[0], b[0]) and np.allclose(a[1], b[1]), "fast IoU/Dice differs from original"
    print(f"Equivalence test passed: fast evaluator == original on {n_trials} random cases.")

run_equivalence_test()

  PASS  T1 perfect -> PQ=1
  PASS  T2 IoU==0.5 not TP
  PASS  T3 IoU>0.5 is TP
  PASS  T4 1 GT + 2 preds -> TP=2 FP=0 FN=0
  PASS  T5 2 GT + 1 pred -> TP=2 FP=0 FN=0
  PASS  T6 no preds -> FN=1, PQ=0
  PASS  T7 no GT -> FP=1, PQ=0
All 7 metric tests passed.
Equivalence test passed: fast evaluator == original on 40 random cases.


In [6]:
from pycocotools import mask as mask_utils

# ============================================================
# RLE (pycocotools, Fortran order, 2048x2048)  — kept for the later submission notebook
# ============================================================

def encode_rle(mask):
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle

def decode_rle(rle):
    r = dict(rle)
    if isinstance(r["counts"], str):
        r["counts"] = r["counts"].encode("utf-8")
    return mask_utils.decode(r).astype(bool)

_t = np.zeros((2048, 2048), bool)
_t[100:130, 400:900] = True; _t[1500:1510, 30:80] = True
_rt = decode_rle(encode_rle(_t))
assert _rt.shape == (2048, 2048) and (_rt == _t).all(), "RLE round-trip mismatch"
assert decode_rle(encode_rle(np.zeros((2048, 2048), bool))).sum() == 0
print("RLE round-trip checks passed (2048x2048, incl. empty mask)")

RLE round-trip checks passed (2048x2048, incl. empty mask)


In [7]:
# ============================================================
# SPLIT — date-grouped 80/20 (Notebook 02 A3). Same function, same seed as Notebook 03.
# ============================================================

FN_RE = re.compile(r"^(\d{4})(\d{2})(\d{2})\d{6}([A-Za-z])h\.", re.I)

def create_split(primary_ids, val_frac=0.2, seed=SEED):
    rows = []
    for img_id in primary_ids:
        fname = Path(images_info[img_id]["file_name"]).name
        m = FN_RE.match(fname)
        if not m:
            raise ValueError(f"Unparseable filename: {fname}")
        rows.append({"img_id": img_id, "file": fname,
                     "date": f"{m[1]}-{m[2]}-{m[3]}", "station": m[4].upper()})
    df = pd.DataFrame(rows)
    dates = sorted(df["date"].unique())
    rng = random.Random(seed); rng.shuffle(dates)
    n_val_dates = int(round(len(dates) * val_frac))
    val_dates = set(dates[:n_val_dates])
    df["split"] = df["date"].map(lambda d: "val" if d in val_dates else "train")
    return df

split_df = create_split(primary_ids)
train_ids = split_df.loc[split_df.split == "train", "img_id"].tolist()
val_ids   = split_df.loc[split_df.split == "val", "img_id"].tolist()

if DEBUG_MODE:                       # smoke-test shortcut — must be OFF for Baseline-01
    train_ids = train_ids[:CFG["debug_n_train"]]
    val_ids   = val_ids[:CFG["debug_n_val"]]

SPLIT_HASH = hashlib.sha1(",".join(sorted(
    split_df.loc[split_df.split == "val", "file"])).encode()).hexdigest()[:12]

split_df.to_csv(OUT / "split_metadata.csv", index=False)
print(f"train: {len(train_ids)} | val: {len(val_ids)} (date-grouped) | split hash: {SPLIT_HASH}")
assert not set(split_df[split_df.split == "train"].date) & \
           set(split_df[split_df.split == "val"].date)

train: 568 | val: 139 (date-grouped) | split hash: 0cc2d729a5b1


In [8]:
import torchvision
import torch.nn.functional as F

# ============================================================
# MODEL — plain U-Net (base 32), UNCHANGED.  Loss — BCE + soft Dice, UNCHANGED
# (logits are cast to float32 inside the loss so AMP cannot affect the loss math).
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, cin=1, base=32):
        super().__init__()
        self.e1, self.e2 = ConvBlock(cin, base), ConvBlock(base, base*2)
        self.e3, self.e4 = ConvBlock(base*2, base*4), ConvBlock(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.mid = ConvBlock(base*8, base*16)
        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2); self.d4 = ConvBlock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2);  self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2);  self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2);    self.d1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        m = self.mid(self.pool(e4))
        d = self.d4(torch.cat([self.u4(m), e4], 1))
        d = self.d3(torch.cat([self.u3(d), e3], 1))
        d = self.d2(torch.cat([self.u2(d), e2], 1))
        d = self.d1(torch.cat([self.u1(d), e1], 1))
        return self.out(d)

ENCODER_SPECS = {
    "resnet34":  {"ctor": torchvision.models.resnet34,  "weights": torchvision.models.ResNet34_Weights.IMAGENET1K_V1,  "ch": [64, 64, 128, 256, 512]},
    "resnet50":  {"ctor": torchvision.models.resnet50,  "weights": torchvision.models.ResNet50_Weights.IMAGENET1K_V1,  "ch": [64, 256, 512, 1024, 2048]},
    "resnet101": {"ctor": torchvision.models.resnet101, "weights": torchvision.models.ResNet101_Weights.IMAGENET1K_V1, "ch": [64, 256, 512, 1024, 2048]},
}

class ResNetUNet(nn.Module):
    """U-Net with a ResNet-34 encoder (torchvision). Input: 1-channel image in [0,1]; the model repeats it to 3 channels and
    applies ImageNet mean/std normalisation internally, so the data pipeline stays unchanged."""
    def __init__(self, pretrained=False, encoder=None):
        super().__init__()
        spec = ENCODER_SPECS[encoder or CFG["encoder"]]; c = spec["ch"]
        weights = spec["weights"] if pretrained else None
        try:
            r = spec["ctor"](weights=weights)
        except Exception as e:
            raise RuntimeError("Could not download the ImageNet encoder weights. Turn Internet ON "
                               "(Session options -> Internet) and run again. Original error: " + repr(e))
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)                 # stride 2,  64 ch
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4   # /4:64  /8:128  /16:256  /32:512
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.d4 = ConvBlock(c[4] + c[3], 256)
        self.d3 = ConvBlock(256 + c[2], 128)
        self.d2 = ConvBlock(128 + c[1], 64)
        self.d1 = ConvBlock(64 + c[0], 64)
        self.d0 = ConvBlock(64, 32)
        self.out = nn.Conv2d(32, 1, 1)

    def encoder_parameters(self):
        for mod in (self.stem, self.l1, self.l2, self.l3, self.l4):
            yield from mod.parameters()

    def decoder_parameters(self):
        for mod in (self.d4, self.d3, self.d2, self.d1, self.d0, self.out):
            yield from mod.parameters()

    def forward(self, x):
        x3 = (x.repeat(1, 3, 1, 1) - self.mean) / self.std
        s0 = self.stem(x3)                      # /2
        s1 = self.l1(self.pool(s0))             # /4
        s2 = self.l2(s1)                        # /8
        s3 = self.l3(s2)                        # /16
        s4 = self.l4(s3)                        # /32
        up = lambda t, ref: F.interpolate(t, size=ref.shape[-2:], mode="bilinear", align_corners=False)
        d = self.d4(torch.cat([up(s4, s3), s3], 1))
        d = self.d3(torch.cat([up(d, s2), s2], 1))
        d = self.d2(torch.cat([up(d, s1), s1], 1))
        d = self.d1(torch.cat([up(d, s0), s0], 1))
        d = self.d0(F.interpolate(d, size=x.shape[-2:], mode="bilinear", align_corners=False))
        return self.out(d)

def build_model(pretrained=False, encoder=None):
    return ResNetUNet(pretrained=pretrained, encoder=encoder).to(DEVICE)

def build_optimizer(model):
    groups = [{"params": list(model.encoder_parameters()), "lr": CFG["lr_encoder"], "base_lr": CFG["lr_encoder"]},
              {"params": list(model.decoder_parameters()), "lr": CFG["lr"], "base_lr": CFG["lr"]}]
    return torch.optim.Adam(groups, weight_decay=CFG["weight_decay"])

def bce_dice_loss(logits, target, eps=1.0):
    logits = logits.float()
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    inter = (prob * target).sum(dim=(2, 3))
    dice = (2 * inter + eps) / (prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + eps)
    return bce + (1 - dice).mean()

USE_AMP = bool(CFG["use_amp"]) and DEVICE.type == "cuda"

def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP)

def build_scaler():
    return torch.amp.GradScaler("cuda", enabled=USE_AMP)

_tmp_model = build_model()
N_PARAMS = sum(p.numel() for p in _tmp_model.parameters())
del _tmp_model
print(f"parameters: {N_PARAMS:,} | AMP: {USE_AMP}")

parameters: 24,522,785 | AMP: True


In [9]:
# ============================================================
# HELPERS: pooled-PQ summary, BN re-estimation, flip-TTA prediction, post-processing
# ============================================================
def summarize(items):
    tp = sum(r["TP"] for r in items); fp = sum(r["FP"] for r in items); fn = sum(r["FN"] for r in items)
    ious = [x for r in items for x in r["tp_ious"]]
    denom = tp + 0.5 * fp + 0.5 * fn
    return {"PQ_pooled": (sum(ious) / denom) if denom else 0.0,
            "PQ_mean_per_image": float(np.mean([r["PQ"] for r in items])) if items else 0.0,
            "TP": tp, "FP": fp, "FN": fn, "mean_tp_iou": float(np.mean(ious)) if ious else 0.0}


def load_model(path):
    ck = torch.load(path, map_location="cpu", weights_only=True)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    m = build_model(); m.load_state_dict(sd); m.eval()
    return m

def recalibrate_bn(model, n_batches=60):
    """Re-estimate BatchNorm running statistics on ~240 training images (weights untouched)."""
    m = copy.deepcopy(model)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.reset_running_stats(); mod.momentum = None
    g = torch.Generator(); g.manual_seed(SEED)
    loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"],
                        drop_last=True, generator=g, worker_init_fn=_worker_init)
    it = iter(loader); m.train()
    with torch.no_grad():
        for _ in range(n_batches):
            b = next(it)
            with autocast_ctx():
                m(b["image"].to(DEVICE))
    del it; m.eval()
    return m

def postprocess(prob, thr, min_px, close_r):
    binary = (prob > thr).astype(np.uint8)
    if close_r:
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE,
                                  cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * close_r + 1, 2 * close_r + 1)))
    n, labels = cv2.connectedComponents(binary, connectivity=8)
    areas = np.bincount(labels.ravel(), minlength=n)
    return [labels == k for k in range(1, n) if areas[k] > min_px]

@torch.no_grad()
def predict_views(model, img_path, resolution, tta=True):
    """Returns (plain prob, flip-TTA-averaged prob) at model resolution + original image shape."""
    img = load_image(img_path)
    small = cv2.resize(img, (resolution, resolution), interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(small, [1, 99])
    small = np.clip((small - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)
    x = torch.from_numpy(small)[None, None].to(DEVICE)
    views = [None, [3], [2], [2, 3]] if tta else [None]
    outs = []
    for dims in views:
        xi = x if dims is None else torch.flip(x, dims)
        p = torch.sigmoid(model(xi))
        if dims is not None:
            p = torch.flip(p, dims)
        outs.append(p[0, 0].cpu().numpy())
    return outs[0], np.mean(outs, axis=0), img.shape

def upsample(p, shape):
    h, w = shape
    return cv2.resize(p, (w, h), interpolation=cv2.INTER_LINEAR)

# not used in this notebook (the weights are already BN-fixed); defined only so the shared helper above has no dangling names
train_ds = None
def _worker_init(_):
    cv2.setNumThreads(0)

In [10]:
# ============================================================
# OFFICIAL-PROTOCOL SCORING HELPERS (from Notebook 07): every annotator record of a file is scored separately and pooled
# ============================================================
vsplit = split_df[split_df.img_id.isin(val_ids)]
date_half = {d: "AB"[i % 2] for i, d in enumerate(sorted(vsplit.date.unique()))}
img_half = {r.img_id: date_half[r.date] for r in vsplit.itertuples()}

def records_of(img_id):
    fname = Path(images_info[img_id]["file_name"]).name
    return sorted(file_to_records[fname])

def entry_results(img_id, preds):
    """One result per annotation record of this file (records with no filaments do not exist in the official GT)."""
    out = []
    for rec in records_of(img_id):
        try:
            gt = build_gt_masks(rec)
        except ValueError:
            continue
        out.append(compute_official_pq(gt, preds))
    return out


def eval_config(prob_dict, thr, min_px, close_r, desc="eval"):
    prim, offi, halves = [], [], {"A": [], "B": []}
    for img_id in tqdm(val_ids, desc=desc, leave=False):
        p, shp = prob_dict[img_id]
        preds = postprocess(upsample(p, shp), thr, min_px, close_r)
        prim.append(compute_official_pq(build_gt_masks(img_id), preds))
        er = entry_results(img_id, preds)
        offi += er; halves[img_half[img_id]] += er
    return {"primary": summarize(prim), "official": summarize(offi),
            "official_A": summarize(halves["A"]), "official_B": summarize(halves["B"])}

In [11]:
# ============================================================
# LOAD THE MEMBERS (BN-fixed weights from the B03 and B04 outputs) AND CACHE VALIDATION PROBABILITIES (flip-TTA averaged)
# ============================================================
MEMBER_SPECS = [("B03_best", "B03_resnet34_unet", "baseline_best.pth",        "resnet34"),
                ("B03_last", "B03_resnet34_unet", "last_epoch_bn_fixed.pth", "resnet34"),
                ("B04_best", "B04_resnet50_s0",   "baseline_best.pth",        "resnet50"),
                ("B04_last", "B04_resnet50_s0",   "last_epoch_bn_fixed.pth", "resnet50")]

def find_file(folder, fname):
    hits = [p for p in Path("/kaggle/input").rglob(fname) if p.parent.name == folder]
    if not hits:
        raise FileNotFoundError(f"{folder}/{fname} not found under /kaggle/input. Add the finished output of that run as an Input "
                                f"(its Output tab must list {fname}).")
    return max(hits, key=lambda p: p.stat().st_mtime)

MEMBERS = {}
for name, folder, fname, enc in MEMBER_SPECS:
    path = find_file(folder, fname)
    mdl = build_model(pretrained=False, encoder=enc)
    mdl.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True)); mdl.eval()
    MEMBERS[name] = mdl
    print(f"{name:<9} {enc:<9} <- {path}")

PROB = {n: {} for n in MEMBERS}
for img_id in tqdm(val_ids, desc="validation probabilities (4 models x flip TTA)"):
    fn = Path(images_info[img_id]["file_name"]).name
    for n, mdl in MEMBERS.items():
        _, tta_p, shp = predict_views(mdl, name_to_path[fn], CFG["train_resolution"], tta=True)
        PROB[n][img_id] = (tta_p, shp)

def combo_probs(names):
    return {i: (np.mean([PROB[n][i][0] for n in names], axis=0), PROB[names[0]][i][1]) for i in val_ids}

B03_best  resnet34  <- /kaggle/input/notebooks/parsabehjattabrizi/notebook-09-b03/B03_resnet34_unet/baseline_best.pth
B03_last  resnet34  <- /kaggle/input/notebooks/parsabehjattabrizi/notebook-09-b03/B03_resnet34_unet/last_epoch_bn_fixed.pth
B04_best  resnet50  <- /kaggle/input/notebooks/parsabehjattabrizi/notebook-10-b04/B04_resnet50_s0/baseline_best.pth
B04_last  resnet50  <- /kaggle/input/notebooks/parsabehjattabrizi/notebook-10-b04/B04_resnet50_s0/last_epoch_bn_fixed.pth


validation probabilities (4 models x flip TTA):   0%|          | 0/139 [00:00<?, ?it/s]

In [12]:
# ============================================================
# MEMBERS AND ENSEMBLES UNDER THE OFFICIAL PROTOCOL (current recipe: flip TTA already applied, threshold 0.5, min 400 px, closing 10)
# ============================================================
R = CFG["recipe"]
CANDS = {"B03_best": ["B03_best"], "B03_last": ["B03_last"], "B04_best": ["B04_best"], "B04_last": ["B04_last"],
         "B03+B04 (best)": ["B03_best", "B04_best"], "B03+B04 (last)": ["B03_last", "B04_last"],
         "all four": ["B03_best", "B03_last", "B04_best", "B04_last"]}
rows = []
for cname, names in CANDS.items():
    res = eval_config(combo_probs(names), R["threshold"], R["min_px"], R["closing_r"], cname)
    o = res["official"]
    rows.append({"candidate": cname, "official_PQ": o["PQ_pooled"], "PQ_A": res["official_A"]["PQ_pooled"], "PQ_B": res["official_B"]["PQ_pooled"],
                 "primary_PQ": res["primary"]["PQ_pooled"], "TP": o["TP"], "FP": o["FP"], "FN": o["FN"], "mean_tp_iou": o["mean_tp_iou"]})
cand_df = pd.DataFrame(rows).sort_values("official_PQ", ascending=False).reset_index(drop=True)
cand_df.to_csv(OUT / "candidates.csv", index=False)
print(cand_df.round(4).to_string(index=False))
print("\nReferences (single models, same recipe): B03 0.4333 | B04 0.4329")
BEST_CAND = cand_df.iloc[0].candidate
print("Best candidate:", BEST_CAND, "| official-protocol validation PQ", round(float(cand_df.iloc[0].official_PQ), 4))

B03_best:   0%|          | 0/139 [00:00<?, ?it/s]

B03_last:   0%|          | 0/139 [00:00<?, ?it/s]

B04_best:   0%|          | 0/139 [00:00<?, ?it/s]

B04_last:   0%|          | 0/139 [00:00<?, ?it/s]

B03+B04 (best):   0%|          | 0/139 [00:00<?, ?it/s]

B03+B04 (last):   0%|          | 0/139 [00:00<?, ?it/s]

all four:   0%|          | 0/139 [00:00<?, ?it/s]

     candidate  official_PQ   PQ_A   PQ_B  primary_PQ   TP  FP  FN  mean_tp_iou
      B04_last       0.4435 0.4526 0.4340      0.4368 1063 577 544       0.6774
      B04_best       0.4419 0.4510 0.4325      0.4355 1064 590 543       0.6772
      all four       0.4394 0.4455 0.4331      0.4334 1067 615 540       0.6773
B03+B04 (best)       0.4389 0.4462 0.4313      0.4345 1065 620 542       0.6783
B03+B04 (last)       0.4365 0.4437 0.4289      0.4309 1061 625 546       0.6773
      B03_last       0.4255 0.4323 0.4183      0.4169 1043 666 564       0.6764
      B03_best       0.4249 0.4339 0.4156      0.4188 1043 675 564       0.6773

References (single models, same recipe): B03 0.4333 | B04 0.4329
Best candidate: B04_last | official-protocol validation PQ 0.4435


In [13]:
# ============================================================
# POST-PROCESSING RE-TUNE on the best candidate (split-half protocol)
# ============================================================
THR_G, MIN_G, CLOSE_G = [0.4, 0.5, 0.6], [200, 300, 400, 600, 800], [0, 5, 10, 15]
src = combo_probs(CANDS[BEST_CAND])
GR = {(t, m_, c): [] for t in THR_G for m_ in MIN_G for c in CLOSE_G}
for img_id in tqdm(val_ids, desc="post-processing grid"):
    p, shp = src[img_id]; prob = upsample(p, shp)
    for t in THR_G:
        binary = (prob > t).astype(np.uint8)
        for c in CLOSE_G:
            bm = binary if c == 0 else cv2.morphologyEx(
                binary, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * c + 1, 2 * c + 1)))
            n, labels = cv2.connectedComponents(bm, connectivity=8)
            areas = np.bincount(labels.ravel(), minlength=n)
            comps = [labels == k for k in range(1, n)]
            for m_ in MIN_G:
                preds = [mk for k, mk in enumerate(comps, 1) if areas[k] > m_]
                GR[(t, m_, c)].append((img_half[img_id], entry_results(img_id, preds)))
            del comps

rows = []
for (t, m_, c), lst in GR.items():
    allr = [r for _, er in lst for r in er]
    A = [r for h, er in lst if h == "A" for r in er]; B = [r for h, er in lst if h == "B" for r in er]
    rows.append({"threshold": t, "min_px": m_, "closing_r": c, "PQ_official": summarize(allr)["PQ_pooled"],
                 "PQ_A": summarize(A)["PQ_pooled"], "PQ_B": summarize(B)["PQ_pooled"]})
grid = pd.DataFrame(rows).sort_values("PQ_official", ascending=False).reset_index(drop=True)
grid.to_csv(OUT / "ensemble_grid.csv", index=False)
print(grid.round(4).head(8).to_string(index=False))

d = grid[(grid.threshold == R["threshold"]) & (grid.min_px == R["min_px"]) & (grid.closing_r == R["closing_r"])].iloc[0]
best = grid.iloc[0]
switch = (best.PQ_A - d.PQ_A >= 0.005) and (best.PQ_B - d.PQ_B >= 0.005)
FINAL_RECIPE = {"threshold": R["threshold"], "min_px": R["min_px"], "closing_r": R["closing_r"]}
if switch:
    FINAL_RECIPE = {"threshold": float(best.threshold), "min_px": int(best.min_px), "closing_r": int(best.closing_r)}
print(f"\ncurrent recipe PQ {float(d.PQ_official):.4f} | grid best {float(best.PQ_official):.4f}")
print("FINAL RECIPE:", FINAL_RECIPE, "(switched)" if switch else "(current recipe kept)")
FINAL_VAL = float(best.PQ_official) if switch else float(d.PQ_official)

post-processing grid:   0%|          | 0/139 [00:00<?, ?it/s]

 threshold  min_px  closing_r  PQ_official   PQ_A   PQ_B
       0.5     400          5       0.4450 0.4516 0.4380
       0.5     400         10       0.4435 0.4526 0.4340
       0.5     400          0       0.4419 0.4510 0.4323
       0.6     400         10       0.4409 0.4530 0.4283
       0.5     600          5       0.4408 0.4454 0.4359
       0.5     600         10       0.4408 0.4474 0.4338
       0.5     300          5       0.4399 0.4447 0.4348
       0.4     400          0       0.4397 0.4463 0.4328

current recipe PQ 0.4435 | grid best 0.4450
FINAL RECIPE: {'threshold': 0.5, 'min_px': 400, 'closing_r': 10} (current recipe kept)


In [14]:
# ============================================================
# TEST SET -> submission_E01_ensemble.csv
# ============================================================
names = CANDS[BEST_CAND]
rows, counts = [], []
for path in tqdm(test_paths, desc="test inference (ensemble)"):
    probs = []
    for n in names:
        _, tta_p, shp = predict_views(MEMBERS[n], path, CFG["train_resolution"], tta=True)
        probs.append(tta_p)
    prob = upsample(np.mean(probs, axis=0), shp)
    preds = postprocess(prob, FINAL_RECIPE["threshold"], FINAL_RECIPE["min_px"], FINAL_RECIPE["closing_r"])
    counts.append(len(preds))
    for k, m_ in enumerate(preds):
        rows.append({"filament_id": f"{path.stem}_{k}", "segmentation_rle": encode_rle(m_)["counts"]})

sub = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
SUB_PATH = Path(OUTPUT_DIR) / "submission_E01_ensemble.csv"
sub.to_csv(SUB_PATH, index=False); sub.to_csv(OUT / SUB_PATH.name, index=False)
chk = pd.read_csv(SUB_PATH)
assert list(chk.columns) == ["filament_id", "segmentation_rle"] and chk.filament_id.is_unique and chk.segmentation_rle.notna().all()
assert set(chk.filament_id.str.split("_", n=1).str[0]) <= {p.stem for p in test_paths}
n_img = int((np.array(counts) > 0).sum())

L = []; add = L.append
add("NOTEBOOK 11 SUMMARY — ensemble of B03 + B04")
add("-" * 70)
add("candidates (official-protocol validation PQ with current recipe):")
for r in cand_df.itertuples():
    add(f"   {r.candidate:<16} {r.official_PQ:.4f}  (halves A {r.PQ_A:.4f} / B {r.PQ_B:.4f}; TP {r.TP} FP {r.FP} FN {r.FN})")
add(f"best candidate: {BEST_CAND} | members: {names}")
add(f"final recipe: {FINAL_RECIPE} | validation PQ {FINAL_VAL:.4f} | references B03 0.4333, B04 0.4329")
add(f"submission: {SUB_PATH.name} — {len(sub)} filaments over {n_img}/{len(test_paths)} test images (mean {np.mean(counts):.1f}/image)")
add("CAVEAT: validation PQ is optimistic (the same split selected checkpoints/candidates); leaderboard has been ~0.10 below validation.")
rep = "\n".join(L); print(rep); (OUT / "notebook11_report.txt").write_text(rep)
json.dump({"candidate": BEST_CAND, "members": names, "recipe": FINAL_RECIPE}, open(OUT / "ensemble_recipe.json", "w"), indent=2)
print("\nSaved to", OUT, ":", sorted(p.name for p in OUT.iterdir()))

test inference (ensemble):   0%|          | 0/180 [00:00<?, ?it/s]

NOTEBOOK 11 SUMMARY — ensemble of B03 + B04
----------------------------------------------------------------------
candidates (official-protocol validation PQ with current recipe):
   B04_last         0.4435  (halves A 0.4526 / B 0.4340; TP 1063 FP 577 FN 544)
   B04_best         0.4419  (halves A 0.4510 / B 0.4325; TP 1064 FP 590 FN 543)
   all four         0.4394  (halves A 0.4455 / B 0.4331; TP 1067 FP 615 FN 540)
   B03+B04 (best)   0.4389  (halves A 0.4462 / B 0.4313; TP 1065 FP 620 FN 542)
   B03+B04 (last)   0.4365  (halves A 0.4437 / B 0.4289; TP 1061 FP 625 FN 546)
   B03_last         0.4255  (halves A 0.4323 / B 0.4183; TP 1043 FP 666 FN 564)
   B03_best         0.4249  (halves A 0.4339 / B 0.4156; TP 1043 FP 675 FN 564)
best candidate: B04_last | members: ['B04_last']
final recipe: {'threshold': 0.5, 'min_px': 400, 'closing_r': 10} | validation PQ 0.4435 | references B03 0.4333, B04 0.4329
submission: submission_E01_ensemble.csv — 1245 filaments over 177/180 test images (mea